<a href="https://colab.research.google.com/github/Soujanya-eng/sentiment_analysis_using_RNN_and_logistic_regression/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [89]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [170]:
import os
print(os.listdir(path))

['IMDB Dataset.csv']


In [171]:
import pandas as pd
import os
csv_path=os.path.join(path,"IMDB Dataset.csv")
df=pd.read_csv(csv_path)

In [92]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [75]:
df.shape

(50000, 2)

In [93]:
df=df.sample(n=4000,random_state=42)
df.head()

,review,sentiment
33553,I really liked this Summerslam due to the look...,positive
9427,Not many television shows appeal to quite as m...,positive
199,The film quickly gets to a major chase scene w...,negative
12447,Jane Austen would definitely approve of this o...,positive
39489,Expectations were somewhat high for me when I ...,negative


In [82]:
df.shape

<class 'pandas.core.frame.DataFrame'>
Index: 4000 entries, 33553 to 24975
Empty DataFrame


In [98]:
df["sentiment"].value_counts()

,count
sentiment,
positive,2028
negative,1972


In [99]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [117]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [103]:
# stop wors for example:is,for,the
stop_words=set(stopwords.words("english"))
stemmer=PorterStemmer()  #removing ing,ed playing,played,plays==>play

In [106]:
# cleaning function
def clean_text(text):
  text=re.sub(r'<.?>','',text)
  text=text.lower()
  text=re.sub(r"[^a-zA-z]",' ',text)
  words=text.split()
  words=[
      stemmer.stem(word)
      for word in words
      if word not in stop_words
  ]
  return " ".join(words)

In [107]:
df["review"]=df["review"].apply(clean_text)

In [108]:
df["sentiment"]=df["sentiment"].map({
    "positive":1,
    "negative":0
})

tokenization

In [149]:
tokenizer=Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df["review"])

In [150]:
X=tokenizer.texts_to_sequences(df["review"])
X=pad_sequences(
    X,
    maxlen=20,
    padding="post"
)
y=df["sentiment"]

In [151]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [158]:
from tensorflow.keras.layers import LSTM,Dropout
model=Sequential()
model.add(
    Embedding(
        input_dim=10000,
        output_dim=64,
        input_length=200
    )
)
model.add(LSTM(
    32,
    dropout=0.2,
    recurrent_dropout=0.2
))

model.add(Dropout(0.5))
model.add(Dense(1,activation="sigmoid"))

In [159]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=["accuracy"]
)

In [160]:
history=model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test)
)

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.6313 - loss: 0.6544 - val_accuracy: 0.7200 - val_loss: 0.5591
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.8288 - loss: 0.4181 - val_accuracy: 0.7688 - val_loss: 0.4934
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.9209 - loss: 0.2382 - val_accuracy: 0.7362 - val_loss: 0.6507
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.9600 - loss: 0.1326 - val_accuracy: 0.7387 - val_loss: 0.7027
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9759 - loss: 0.0866 - val_accuracy: 0.7150 - val_loss: 0.8405
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9881 - loss: 0.0511 - val_accuracy: 0.7237 - val_loss: 0.9956
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9919 - loss: 0.0327 - val_accuracy: 0.7038 - val_loss: 1.2610
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9922 - loss: 0.0299 - val_accu

In [161]:
loss,accuracy=model.evaluate(X_test,y_test)
print("loss:",loss)
print("accuracy:",accuracy)

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7237 - loss: 1.1883
loss: 1.188339114189148
accuracy: 0.7237499952316284


In [164]:
review =input("enter the review:")

review = clean_text(review[0])

review_seq = tokenizer.texts_to_sequences([review])

review_pad = pad_sequences(
    review_seq,
    maxlen=200
)

prediction = model.predict(review_pad)

if prediction[0][0] > 0.5:
    print("Positive")
else:
    print("Negative")

enter the review:it is waste of money
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Positive


In [168]:
reviews = [
    "I love this movie",
    "Amazing film",
    "Excellent acting",
    "Worst movie ever",
    "Terrible experience",
    "Waste of money"
]
for review in reviews:

  review = clean_text(review)

  seq = tokenizer.texts_to_sequences([review])

  pad = pad_sequences(seq, maxlen=200)

  pred = model.predict(pad)

  print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
[[0.99991333]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
[[0.99989194]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
[[0.9999241]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 461ms/step
[[0.9989699]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 514ms/step
[[0.9996146]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
[[0.9926782]]


In [112]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=3000)
X_train=vectorizer.fit_transform(X_train)
X_test=vectorizer.transform(X_test)

In [113]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [114]:
y_pred = model.predict(X_test)

In [115]:
from sklearn.metrics import accuracy_score,classification_report
print("accuracy_score:",accuracy_score(y_test,y_pred))
print("classification report:",classification_report(y_test,y_pred))

accuracy_score: 0.8675
classification report:               precision    recall  f1-score   support

           0       0.88      0.84      0.86       392
           1       0.86      0.89      0.87       408

    accuracy                           0.87       800
   macro avg       0.87      0.87      0.87       800
weighted avg       0.87      0.87      0.87       800



In [116]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[331  61]
 [ 45 363]]


In [172]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK resources
nltk.download('stopwords')

# Load Dataset
csv_path=os.path.join(path,"IMDB Dataset.csv")
df=pd.read_csv(csv_path)

# Reduce dataset size
df = df.sample(n=4000, random_state=42)

# Stopwords and Stemmer
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Text Cleaning Function
def clean_text(text):

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Convert to lowercase
    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenize
    words = text.split()

    # Remove stopwords and apply stemming
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

# Apply preprocessing
df['review'] = df['review'].apply(clean_text)

# Convert labels
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# Features and Target
X = df['review']
y = df['sentiment']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# Logistic Regression
model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Accuracy: 0.87

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.85      0.87       392
           1       0.86      0.88      0.87       408

    accuracy                           0.87       800
   macro avg       0.87      0.87      0.87       800
weighted avg       0.87      0.87      0.87       800


Confusion Matrix:
[[335  57]
 [ 47 361]]


In [173]:
review =input("enter the review:")

review = clean_text(review)

review_vector = vectorizer.transform([review])

prediction = model.predict(review_vector)

probability = model.predict_proba(review_vector)

print("Prediction:", prediction[0])
print("Probability:", probability)

if prediction[0] == 1:
    print("Positive")
else:
    print("Negative ")

Prediction: 0
Probability: [[0.89350218 0.10649782]]
Negative 😞


This project developed a Sentiment Analysis system using both RNN and TF-IDF with Logistic Regression on a subset of 4000 IMDb movie reviews. After comparing the models, Logistic Regression with TF-IDF achieved a higher accuracy of 86.75%, while the RNN model showed lower accuracy due to overfitting and limited training data.